In [1]:
import pandas as pd
import os

# --- Configuration ---
# !!! IMPORTANT: Verify this path is correct for your system !!!
file_path = "/Users/mauricebaier/Documents/Master/4th_master_thesis/_code/math-reasoning-in-language-models/data/Relationship_Quant_Qual.xlsx"

# !!! IMPORTANT: Verify these column names EXACTLY match your Excel file !!!
# Based on the image, the benchmark column name might be truncated visually.
# Using the name as it appears in the image snippet. Adjust if needed.
benchmark_col = "Benchmark score (Zero Shot)"
human_assessment_cols = ["Coherence", "Context", "Completeness", "Overall"]

# Check if the file exists before proceeding
if not os.path.exists(file_path):
    print(f"Error: File not found at path: {file_path}")
    print("Please ensure the file path is correct.")
    exit()

# --- Load Data ---
try:
    df = pd.read_excel(file_path)
    print(f"Successfully loaded data from: {file_path}")
    # Optional: Print columns found to help debug names if needed
    # print("\nColumns found in the Excel file:")
    # print(df.columns.tolist())

except Exception as e:
    print(f"Error loading Excel file: {e}")
    exit()

# --- Data Validation and Cleaning ---
required_cols = [benchmark_col] + human_assessment_cols
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print("\nError: The following required columns were not found in the Excel file:")
    for col in missing_cols:
        print(f"- '{col}'")
    print("\nPlease check the column names in your Excel file and update the script.")
    print("\nColumns found in the file:")
    print(df.columns.tolist())
    exit()

# Make a copy to avoid SettingWithCopyWarning
df_cleaned = df.copy()

# Clean the 'Benchmark score' column:
# 1. Convert to string (to safely handle potential non-string types)
# 2. Remove the '%' sign
# 3. Convert to numeric (float), coercing errors to NaN
# 4. Divide by 100 to get the decimal representation
try:
    df_cleaned[benchmark_col] = df_cleaned[benchmark_col].astype(str).str.replace('%', '', regex=False)
    df_cleaned[benchmark_col] = pd.to_numeric(df_cleaned[benchmark_col], errors='coerce') / 100.0

    # Check if any NaNs were created during conversion
    if df_cleaned[benchmark_col].isnull().any():
        print(f"\nWarning: Some values in '{benchmark_col}' could not be converted to numeric format after removing '%'.")
        print("Rows with conversion issues (showing original data):")
        print(df[df_cleaned[benchmark_col].isnull()])
        # Decide how to handle NaNs: drop rows or imputation might be needed
        # For correlation, pandas usually handles NaNs by pairwise deletion by default.

except Exception as e:
    print(f"\nError cleaning benchmark column '{benchmark_col}': {e}")
    exit()

# Convert human assessment columns to numeric, coercing errors
for col in human_assessment_cols:
    original_dtype = df_cleaned[col].dtype
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
    if df_cleaned[col].isnull().any() and not pd.api.types.is_numeric_dtype(original_dtype):
         print(f"\nWarning: Some values in '{col}' were non-numeric and converted to NaN.")

# --- Calculate Correlations ---
print("\n--- Correlation Analysis ---")

results = {}
cols_to_correlate = [benchmark_col] + human_assessment_cols
correlation_data = df_cleaned[cols_to_correlate].dropna() # Use dropna for listwise deletion (only rows with valid numbers in *all* these columns)
                                                      # Alternatively, calculate one by one for pairwise deletion:
                                                      # e.g., df_cleaned[[benchmark_col, col]].dropna().corr().iloc[0, 1]


if correlation_data.shape[0] < 2:
    print(f"\nError: Not enough valid data points (found {correlation_data.shape[0]}) to calculate correlations after handling missing/invalid values.")
    print("Check your data for missing values or conversion errors in the relevant columns.")
else:
    print(f"Calculating correlations using {correlation_data.shape[0]} complete rows (listwise deletion).")

    # --- Pearson Correlation (measures linear relationship) ---
    print("\nPearson Correlation Coefficients:")
    try:
        pearson_corr = correlation_data.corr(method='pearson')
        # Extract correlations with the benchmark score
        pearson_results = pearson_corr[benchmark_col].drop(benchmark_col) # Drop self-correlation
        results['Pearson'] = pearson_results
        for col, corr_value in pearson_results.items():
            print(f"- '{benchmark_col}' vs '{col}': {corr_value:.4f}")
    except Exception as e:
        print(f"Error calculating Pearson correlation: {e}")


    # --- Spearman Correlation (measures monotonic relationship, less sensitive to outliers) ---
    print("\nSpearman Correlation Coefficients:")
    try:
        spearman_corr = correlation_data.corr(method='spearman')
        # Extract correlations with the benchmark score
        spearman_results = spearman_corr[benchmark_col].drop(benchmark_col) # Drop self-correlation
        results['Spearman'] = spearman_results
        for col, corr_value in spearman_results.items():
            print(f"- '{benchmark_col}' vs '{col}': {corr_value:.4f}")
    except Exception as e:
        print(f"Error calculating Spearman correlation: {e}")

# --- Optional: Display results as a DataFrame ---
if results:
    results_df = pd.DataFrame(results)
    print("\n--- Summary Table ---")
    print(results_df)

Successfully loaded data from: /Users/mauricebaier/Documents/Master/4th_master_thesis/_code/math-reasoning-in-language-models/data/Relationship_Quant_Qual.xlsx

--- Correlation Analysis ---
Calculating correlations using 14 complete rows (listwise deletion).

Pearson Correlation Coefficients:
- 'Benchmark score (Zero Shot)' vs 'Coherence': 0.8883
- 'Benchmark score (Zero Shot)' vs 'Context': 0.9084
- 'Benchmark score (Zero Shot)' vs 'Completeness': 0.8566
- 'Benchmark score (Zero Shot)' vs 'Overall': 0.8878

Spearman Correlation Coefficients:
- 'Benchmark score (Zero Shot)' vs 'Coherence': 0.6556
- 'Benchmark score (Zero Shot)' vs 'Context': 0.7938
- 'Benchmark score (Zero Shot)' vs 'Completeness': 0.6191
- 'Benchmark score (Zero Shot)' vs 'Overall': 0.6350

--- Summary Table ---
               Pearson  Spearman
Coherence     0.888263  0.655629
Context       0.908418  0.793826
Completeness  0.856639  0.619051
Overall       0.887789  0.634957


In [ ]:
import pandas as pd
import os

# --- Configuration ---
# !!! IMPORTANT: Verify this path is correct for your system !!!
file_path = "/Users/mauricebaier/Documents/Master/4th_master_thesis/_code/math-reasoning-in-language-models/data/Relationship_Quant_Qual.xlsx"

# !!! IMPORTANT: Verify these column names EXACTLY match your Excel file !!!
# Based on the image, the benchmark column name might be truncated visually.
# Using the name as it appears in the image snippet. Adjust if needed.
benchmark_col = "Benchmark score (Zero Shot)"
human_assessment_cols = ["Coherence", "Context", "Completeness", "Overall"]

# Check if the file exists before proceeding
if not os.path.exists(file_path):
    print(f"Error: File not found at path: {file_path}")
    print("Please ensure the file path is correct.")
    exit()

# --- Load Data ---
try:
    df = pd.read_excel(file_path)
    print(f"Successfully loaded data from: {file_path}")
    # Optional: Print columns found to help debug names if needed
    # print("\nColumns found in the Excel file:")
    # print(df.columns.tolist())

except Exception as e:
    print(f"Error loading Excel file: {e}")
    exit()

# --- Data Validation and Cleaning ---
required_cols = [benchmark_col] + human_assessment_cols
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print("\nError: The following required columns were not found in the Excel file:")
    for col in missing_cols:
        print(f"- '{col}'")
    print("\nPlease check the column names in your Excel file and update the script.")
    print("\nColumns found in the file:")
    print(df.columns.tolist())
    exit()

# Make a copy to avoid SettingWithCopyWarning
df_cleaned = df.copy()

# Clean the 'Benchmark score' column:
# 1. Convert to string (to safely handle potential non-string types)
# 2. Remove the '%' sign
# 3. Convert to numeric (float), coercing errors to NaN
# 4. Divide by 100 to get the decimal representation
try:
    df_cleaned[benchmark_col] = df_cleaned[benchmark_col].astype(str).str.replace('%', '', regex=False)
    df_cleaned[benchmark_col] = pd.to_numeric(df_cleaned[benchmark_col], errors='coerce') / 100.0

    # Check if any NaNs were created during conversion
    if df_cleaned[benchmark_col].isnull().any():
        print(f"\nWarning: Some values in '{benchmark_col}' could not be converted to numeric format after removing '%'. These rows will be excluded pair-wise during correlation.")
        # print("Rows with conversion issues (showing original data):") # Optional detail
        # print(df[df_cleaned[benchmark_col].isnull()])                 # Optional detail

except Exception as e:
    print(f"\nError cleaning benchmark column '{benchmark_col}': {e}")
    exit()

# Convert human assessment columns to numeric, coercing errors
for col in human_assessment_cols:
    original_dtype = df_cleaned[col].dtype # Store original type to check if it was already numeric
    df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')
    if df_cleaned[col].isnull().any() and not pd.api.types.is_numeric_dtype(original_dtype):
         print(f"\nWarning: Some values in '{col}' were non-numeric and converted to NaN. These rows will be excluded pair-wise during correlation.")

# --- Calculate Correlations (Pairwise) ---
print("\n--- Correlation Analysis ---")
print(f"Correlating each human assessment score with '{benchmark_col}'")

correlation_results = {}

for human_col in human_assessment_cols:
    print(f"\nCalculating correlation for: '{human_col}'")

    # Select the two columns for pairwise correlation
    # .dropna() here ensures only rows with valid values in BOTH columns are used for this specific pair
    pair_df = df_cleaned[[human_col, benchmark_col]].dropna()

    if pair_df.shape[0] < 2:
        print(f"  - Skipped: Not enough valid data points ({pair_df.shape[0]}) for correlation between '{human_col}' and '{benchmark_col}'.")
        correlation_results[human_col] = {'Pearson': None, 'Spearman': None, 'N': pair_df.shape[0]}
        continue # Skip to the next human column

    # Calculate Pearson Correlation
    try:
        pearson_corr = pair_df.corr(method='pearson').loc[human_col, benchmark_col]
        print(f"  - Pearson Correlation: {pearson_corr:.4f}")
    except Exception as e:
        print(f"  - Error calculating Pearson correlation: {e}")
        pearson_corr = None

    # Calculate Spearman Correlation
    try:
        spearman_corr = pair_df.corr(method='spearman').loc[human_col, benchmark_col]
        print(f"  - Spearman Correlation: {spearman_corr:.4f}")
    except Exception as e:
        print(f"  - Error calculating Spearman correlation: {e}")
        spearman_corr = None

    print(f"  - (Based on N = {pair_df.shape[0]} valid pairs)")
    correlation_results[human_col] = {'Pearson': pearson_corr, 'Spearman': spearman_corr, 'N': pair_df.shape[0]}


# --- Optional: Display results as a DataFrame ---
print("\n--- Summary Table ---")
results_df = pd.DataFrame(correlation_results).T # Transpose to have human scores as rows
print(results_df)

Successfully loaded data from: /Users/mauricebaier/Documents/Master/4th_master_thesis/_code/math-reasoning-in-language-models/data/Relationship_Quant_Qual.xlsx

Error: The following required columns were not found in the Excel file:
- 'Benchmark score (Zero S'

Please check the column names in your Excel file and update the script.

Columns found in the file:
['Model', 'Benchmark score (Zero Shot)', 'Coherence', 'Context', 'Completeness', 'Overall']

Error cleaning benchmark column 'Benchmark score (Zero S': 'Benchmark score (Zero S'

--- Correlation Analysis ---
Correlating each human assessment score with 'Benchmark score (Zero S'

Calculating correlation for: 'Coherence'


KeyError: "['Benchmark score (Zero S'] not in index"

: 